# DANNs on imbalanced data 

# 1. setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, RobustScaler  # ← Add RobustScaler here!
import os 
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

print("="*60)
print("DANN EXPERIMENT: IMBALANCED DATASETS")
print("="*60)

torch.manual_seed(42)
np.random.seed(42)

# Memory check
print(f"Available memory: {psutil.virtual_memory().available / 1024**3:.1f} GB")

# 2. load imbalanced data

In [ ]:
c4_imbalanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_processed.csv')
ybt_imbalanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_processed.csv')

print(f"c4 shape: {c4_imbalanced.shape}")
print(f"ybt shape: {ybt_imbalanced.shape}")

# check class dist
print("c4 class dist:")
print(c4_imbalanced['autism_target'].value_counts())
print(f"c4 class balance: {c4_imbalanced['autism_target'].value_counts(normalize=True)}")

print("\nybt class dist:")
print(ybt_imbalanced['autism_target'].value_counts())
print(f"ybt class balance: {ybt_imbalanced['autism_target'].value_counts(normalize=True)}")

# memory usage after laoding 
print(f"memory usage: {psutil.virtual_memory().percent}%")


# 3. feature engineering (same as balanced)

In [ ]:
print("="*60)
print("FEATURE ENGINEERING FOR IMBALANCED DATA")
print("="*60)

# --- Robust Feature Engineering for Maximum Overlap ---

def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items + 1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_imbalanced = create_aggregate_features(c4_imbalanced, prefix, n_items)
    ybt_imbalanced = create_aggregate_features(ybt_imbalanced, prefix, n_items)

# D-score
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total'] 
    else:
        print(f"d_score not created for this dataframe")

# Age-EQ interaction 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']
    else:
        print(f"age_x_eq not created for this dataframe")

# Age-AQ interaction
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']
    else:
        print(f"age_x_aq not created for this dataframe")

# Age-SQR interaction
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns and 'sqr_total' in df.columns:  # ← CORRECT
        df['age_x_sqr'] = df['age'] * df['sqr_total']

# AQ-EQ interaction 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    else:
        print(f"aq_eq_interaction not created for this dataframe")

# EQ/SQR ratio
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    else:
        print(f"eq_sqr_ratio not created for this dataframe")

# Log-transformed AQ total 
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))

# Square root of age
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High AQ flag
for df in [c4_imbalanced, ybt_imbalanced]:
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Now recompute feature lists
exclude_cols = ['autism_target']  # Only exclude target

# Add these only if they exist
if 'userid' in c4_imbalanced.columns:
    exclude_cols.append('userid')
if 'date' in c4_imbalanced.columns:
    exclude_cols.append('date')
if 'timestamp' in c4_imbalanced.columns:
    exclude_cols.append('timestamp')

c4_features = [col for col in c4_imbalanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_imbalanced.columns if col not in exclude_cols]

common_features = sorted(list(set(c4_features) & set(ybt_features)))
print(f"Common features after alignment: {len(common_features)}")
print("Sample common features:", common_features[:5])

# Print missing features 
missing_in_ybt = set(c4_features) - set(ybt_features)
missing_in_c4 = set(ybt_features) - set(c4_features)
print(f"Features missing in YBT: {len(missing_in_ybt)}")
print(f"Features in YBT but missing in C4: {len(missing_in_c4)}")

# Memory check
print(f"Memory usage: {psutil.virtual_memory().percent}%")

# 4. data preperation (imbalanced)

In [ ]:
print("DATA PREPERATION: IMBALANCED DATA")

# recreate data arrays with new features
x_c4_imb = c4_imbalanced[common_features].values
y_c4_imb = c4_imbalanced['autism_target'].values
x_ybt_imb = ybt_imbalanced[common_features].values
y_ybt_imb = ybt_imbalanced['autism_target'].values

print(f"original c4 shape:{x_c4_imb.shape}")
print(f"original ybt shape:{x_ybt_imb.shape}")

# standardization
scaler_imb = RobustScaler()
x_c4_imb_scaled = scaler_imb.fit_transform(x_c4_imb)
x_ybt_imb_scaled = scaler_imb.transform(x_ybt_imb)

# split c4 imbalanced data 
x_train_imb, x_val_imb, y_train_imb, y_val_imb = train_test_split(
    x_c4_imb_scaled, y_c4_imb, test_size=0.2, random_state=42, stratify=y_c4_imb
)

print(f"X_train_imb shape: {x_train_imb.shape}")
print(f"X_val_imb shape: {x_val_imb.shape}")
print(f"X_ybt_imb_scaled shape: {x_ybt_imb_scaled.shape}")

print(f"Train class distribution: {np.bincount(y_train_imb)}")
print(f"val class distribution: {np.bincount(y_val_imb)}")
print(f"YBT class distribution: {np.bincount(y_ybt_imb)}")

print("NaNs in x_train_imb:", np.isnan(x_train_imb).sum())
print("infs in x_train_imb:", np.isinf(x_train_imb).sum())

# memory usage 
print(f"memory usage: {psutil.virtual_memory().percent}%")

# 5. gradient reversal layer

In [ ]:
print("\n" + "="*60)
print("GRADIENT REVERSAL LAYER")
print("="*60)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# 6. DANN model 

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN ADVERSARIAL CLASSIFIER (IMBALANCED)")
print("="*60)

class ImprovedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, alpha=1.0, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
        
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 1),
            nn.Sigmoid()
        )
        
        # Domain discriminator
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        self.class_criterion = nn.BCELoss()
        self.domain_criterion = nn.BCELoss()
        self.domain_weight = domain_weight

    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output

    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        
        self.log('train_loss', total_loss)
        self.log('train_class_loss', class_loss)
        self.log('train_domain_loss', domain_loss)
        self.log('train_f1', f1)
        
        return total_loss

    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        
        class_output, domain_output = self(x)
        
        # Classification loss
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Domain discrimination loss
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        
        # Total loss
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Metrics
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        
        self.log('val_loss', total_loss)
        self.log('val_class_loss', class_loss)
        self.log('val_domain_loss', domain_loss)
        self.log('val_f1', f1)
        
        return total_loss

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_f1"
            }
        }

# 7. data module

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN ADAPTATION DATA MODULE (IMBALANCED)")
print("="*60)

class ImprovedDomainAdaptationDataModule(L.LightningDataModule):
    def __init__(self, X_train, X_val, y_train, y_val, X_test, y_test, batch_size=32):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.batch_size = batch_size

    def setup(self, stage=None):
        # Create datasets
        self.train_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_train),
            torch.LongTensor(self.y_train),
            torch.zeros(len(self.X_train))  # Domain 0 for C4
        )
        
        self.val_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_val),
            torch.LongTensor(self.y_val),
            torch.zeros(len(self.X_val))  # Domain 0 for C4
        )
        
        self.test_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_test),
            torch.LongTensor(self.y_test),
            torch.ones(len(self.X_test))  # Domain 1 for YBT
        )

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset, batch_size=self.batch_size, shuffle=True,
            num_workers=4, pin_memory=True
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=4, pin_memory=True
        )

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=4, pin_memory=True
        )

# Create data module for imbalanced experiment
data_module_imb = ImprovedDomainAdaptationDataModule(
    x_train_imb, x_val_imb, y_train_imb, y_val_imb, x_ybt_imb_scaled, y_ybt_imb, batch_size=32
)

# 8. train imbalanced DANN

In [ ]:
print("\n" + "="*60)
print("TRAINING IMBALANCED DANN MODEL")
print("="*60)

# Initialize model for imbalanced data
model_imb = ImprovedDomainAdversarialClassifier(
    input_dim=len(common_features),
    hidden_dims=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=0.0005,
    alpha=1.0,
    domain_weight=0.05
)

# Trainer with imbalanced-appropriate settings
trainer_imb = L.Trainer(
    max_epochs=100,
    accelerator='auto',
    devices=1,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(monitor='val_f1', patience=15, mode='max', verbose=True),
        L.pytorch.callbacks.ModelCheckpoint(monitor='val_f1', mode='max', save_top_k=3, verbose=True),
        L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch')
    ],
    log_every_n_steps=25,
    enable_progress_bar=True,
    enable_model_summary=True,
    deterministic=True
)

# Train the model
print("Training imbalanced DANN model...")
trainer_imb.fit(model_imb, data_module_imb)

print("Imbalanced DANN training completed!")
print(f"Best validation F1: {trainer_imb.checkpoint_callback.best_model_score:.3f}")

# 8.5 Defining DANN model

In [ ]:
print("="*60)
print("GETTING EXACT 44 FEATURES FROM BALANCED MODEL")
print("="*60)

# Based on the balanced notebook output, the 44 features included:
# Sample common features: ['age', 'age_x_aq', 'age_x_eq', 'aq_1', 'aq_10', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6']

# Let's check what features are actually available in your imbalanced data
print(f"Your imbalanced data has {len(common_features_44)} features after engineering:")
for i, feat in enumerate(sorted(common_features_44)):
    print(f"{i+1:2d}. {feat}")

# The balanced model had 44 features, but we need to find what the other 2 were
# Let's check if there are any features in your imbalanced data that we're not using
all_c4_features = [col for col in c4_imbalanced.columns if col not in ['autism_target', 'userid', 'date', 'timestamp']]
all_ybt_features = [col for col in ybt_imbalanced.columns if col not in ['autism_target', 'userid', 'date', 'timestamp']]

print(f"\nAll C4 features: {len(all_c4_features)}")
print(f"All YBT features: {len(all_ybt_features)}")

# Find features that are in both datasets but not in our current list
common_all = sorted(list(set(all_c4_features) & set(all_ybt_features)))
print(f"All common features: {len(common_all)}")

# Find what we're missing
missing_from_current = [f for f in common_all if f not in available_44]
print(f"\nFeatures we're missing: {missing_from_current}")

if len(missing_from_current) >= 2:
    print(f"\nWe can add these {len(missing_from_current)} features to get to 44!")
    # Add the missing features
    for feat in missing_from_current[:2]:  # Take first 2 to get to 44
        if feat in c4_imbalanced.columns and feat in ybt_imbalanced.columns:
            print(f"Adding feature: {feat}")
            available_44.append(feat)
else:
    print(f"\nWe only have {len(missing_from_current)} additional features, need {2 - len(missing_from_current)} more")

# Use exactly 44 features
final_44_features = available_44[:44]  # Take first 44
print(f"\nFinal 44 features: {len(final_44_features)}")

# Recreate data arrays with exactly 44 features
x_c4_imb_final = c4_imbalanced[final_44_features].values
x_ybt_imb_final = ybt_imbalanced[final_44_features].values

# Re-standardize
scaler_final = RobustScaler()
x_c4_imb_scaled_final = scaler_final.fit_transform(x_c4_imb_final)
x_ybt_imb_scaled_final = scaler_final.transform(x_ybt_imb_final)

print(f"Final shapes with exactly 44 features:")
print(f"x_c4_imb_scaled_final: {x_c4_imb_scaled_final.shape}")
print(f"x_ybt_imb_scaled_final: {x_ybt_imb_scaled_final.shape}")

In [ ]:
print("="*60)
print("FIXING FEATURE ENGINEERING MISMATCH")
print("="*60)

# The issue is that your imbalanced data has different feature engineering
# Let's recreate the EXACT same features as the balanced model

# First, let's see what the balanced model actually used
print("Loading balanced data to see exact features...")
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

# Apply EXACT same feature engineering as balanced notebook
def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

# Apply EXACT same engineering to balanced data
for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_balanced = create_aggregate_features(c4_balanced, prefix, n_items)
    ybt_balanced = create_aggregate_features(ybt_balanced, prefix, n_items)

# Apply EXACT same feature engineering as balanced model
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total']
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Get balanced common features
c4_balanced_features = [col for col in c4_balanced.columns if col != 'autism_target']
ybt_balanced_features = [col for col in ybt_balanced.columns if col != 'autism_target']
balanced_common = sorted(list(set(c4_balanced_features) & set(ybt_balanced_features)))

print(f"Balanced common features: {len(balanced_common)}")
print(f"Sample balanced features: {balanced_common[:10]}")

# Now apply EXACT same engineering to imbalanced data
print("\nApplying exact same engineering to imbalanced data...")
c4_imbalanced_fixed = c4_imbalanced.copy()
ybt_imbalanced_fixed = ybt_imbalanced.copy()

# Apply EXACT same feature engineering
for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_imbalanced_fixed = create_aggregate_features(c4_imbalanced_fixed, prefix, n_items)
    ybt_imbalanced_fixed = create_aggregate_features(ybt_imbalanced_fixed, prefix, n_items)

for df in [c4_imbalanced_fixed, ybt_imbalanced_fixed]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total']
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Get imbalanced common features
c4_imb_features = [col for col in c4_imbalanced_fixed.columns if col != 'autism_target']
ybt_imb_features = [col for col in ybt_imbalanced_fixed.columns if col != 'autism_target']
imb_common = sorted(list(set(c4_imb_features) & set(ybt_imb_features)))

print(f"Imbalanced common features: {len(imb_common)}")

# Find the intersection of balanced and imbalanced features
final_features = sorted(list(set(balanced_common) & set(imb_common)))
print(f"Final common features: {len(final_features)}")

# Use balanced scaler on the correctly engineered features
x_c4_balanced_final = c4_balanced[final_features].values
x_ybt_balanced_final = ybt_balanced[final_features].values
x_c4_imb_final = c4_imbalanced_fixed[final_features].values
x_ybt_imb_final = ybt_imbalanced_fixed[final_features].values

# Fit scaler on balanced data
balanced_scaler_final = RobustScaler()
x_c4_balanced_scaled = balanced_scaler_final.fit_transform(x_c4_balanced_final)
x_ybt_balanced_scaled = balanced_scaler_final.transform(x_ybt_balanced_final)

# Apply to imbalanced data
x_c4_imb_scaled_final = balanced_scaler_final.transform(x_c4_imb_final)
x_ybt_imb_scaled_final = balanced_scaler_final.transform(x_ybt_imb_final)

print(f"Final shapes:")
print(f"x_ybt_imb_scaled_final: {x_ybt_imb_scaled_final.shape}")
print(f"Feature range: [{x_ybt_imb_scaled_final.min():.3f}, {x_ybt_imb_scaled_final.max():.3f}]")
print(f"Feature mean: {x_ybt_imb_scaled_final.mean():.3f}")
print(f"Feature std: {x_ybt_imb_scaled_final.std():.3f}")

In [ ]:
print("="*60)
print("FIXING SCALING ISSUE")
print("="*60)

# The problem is that the imbalanced data has extreme values even after scaling
# Let's clip the values to reasonable ranges first, then scale

# Clip extreme values before scaling
x_c4_imb_clipped = np.clip(x_c4_imb_final, -10, 10)
x_ybt_imb_clipped = np.clip(x_ybt_imb_final, -10, 10)

print(f"After clipping:")
print(f"x_c4_imb_clipped range: [{x_c4_imb_clipped.min():.3f}, {x_c4_imb_clipped.max():.3f}]")
print(f"x_ybt_imb_clipped range: [{x_ybt_imb_clipped.min():.3f}, {x_ybt_imb_clipped.max():.3f}]")

# Now scale with balanced scaler
balanced_scaler_clipped = RobustScaler()
x_c4_balanced_scaled = balanced_scaler_clipped.fit_transform(x_c4_balanced_final)
x_ybt_imb_scaled_clipped = balanced_scaler_clipped.transform(x_ybt_imb_clipped)

print(f"After scaling:")
print(f"x_ybt_imb_scaled_clipped range: [{x_ybt_imb_scaled_clipped.min():.3f}, {x_ybt_imb_scaled_clipped.max():.3f}]")
print(f"x_ybt_imb_scaled_clipped mean: {x_ybt_imb_scaled_clipped.mean():.3f}")
print(f"x_ybt_imb_scaled_clipped std: {x_ybt_imb_scaled_clipped.std():.3f}")

# Test with clipped and properly scaled data
ybt_predictions_clipped = []
ybt_probs_clipped = []
ybt_targets_clipped = []

x_ybt_tensor_clipped = torch.FloatTensor(x_ybt_imb_scaled_clipped)
ybt_dataset_clipped = torch.utils.data.TensorDataset(
    x_ybt_tensor_clipped, torch.LongTensor(y_ybt_imb), torch.ones(len(x_ybt_imb_scaled_clipped))
)
ybt_dataloader_clipped = torch.utils.data.DataLoader(ybt_dataset_clipped, batch_size=32, shuffle=False)

device = next(balanced_model.parameters()).device

with torch.no_grad():
    for batch in ybt_dataloader_clipped:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = balanced_model(x)
        ybt_probs_clipped.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_clipped.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_clipped.extend(y.cpu().numpy())

ybt_probs_clipped = np.array(ybt_probs_clipped)
ybt_predictions_clipped = np.array(ybt_predictions_clipped)
ybt_targets_clipped = np.array(ybt_targets_clipped)

# Calculate metrics
clipped_f1 = f1_score(ybt_targets_clipped, ybt_predictions_clipped, average='weighted')
clipped_auc = roc_auc_score(ybt_targets_clipped, ybt_probs_clipped)

print(f"\nBalanced DANN on imbalanced YBT (CLIPPED):")
print(f"F1 Score: {clipped_f1:.3f}")
print(f"ROC-AUC: {clipped_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_clipped)}")
print(f"Prediction distribution: {np.bincount(ybt_predictions_clipped)}")

# Check prediction probabilities
print(f"Prediction probabilities range: [{ybt_probs_clipped.min():.3f}, {ybt_probs_clipped.max():.3f}]")
print(f"Mean prediction: {ybt_probs_clipped.mean():.3f}")
print(f"Std prediction: {ybt_probs_clipped.std():.3f}")

# 9. test balanced DANN on imbalanced data 

In [ ]:
print("="*60)
print("OPTIMIZING THRESHOLD FOR IMBALANCED DATA")
print("="*60)

# The model is predicting probabilities well (AUC=0.754) but the 0.5 threshold is wrong
# Let's find the optimal threshold for imbalanced data

from sklearn.metrics import precision_recall_curve, f1_score

# Find optimal threshold using precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(ybt_targets_clipped, ybt_probs_clipped)

# Calculate F1 for each threshold
f1_scores = []
for i in range(len(thresholds)):
    f1 = f1_score(ybt_targets_clipped, (ybt_probs_clipped >= thresholds[i]).astype(int), average='weighted')
    f1_scores.append(f1)

# Find best threshold
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best F1 score: {best_f1:.3f}")

# Test with optimal threshold
ybt_predictions_optimal = (ybt_probs_clipped >= best_threshold).astype(int)
optimal_f1 = f1_score(ybt_targets_clipped, ybt_predictions_optimal, average='weighted')

print(f"\nResults with optimal threshold:")
print(f"F1 Score: {optimal_f1:.3f}")
print(f"ROC-AUC: {clipped_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_clipped)}")
print(f"Prediction distribution: {np.bincount(ybt_predictions_optimal)}")

# Compare different thresholds
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print(f"\nF1 scores at different thresholds:")
for thresh in thresholds_to_test:
    preds = (ybt_probs_clipped >= thresh).astype(int)
    f1 = f1_score(ybt_targets_clipped, preds, average='weighted')
    print(f"Threshold {thresh}: F1 = {f1:.3f}")

print(f"\nCONCLUSION:")
print(f"The model CAN distinguish classes (AUC=0.754) but needs a lower threshold")
print(f"for imbalanced data. The default 0.5 threshold is too high.")


print("\n" + "="*60)
print("TESTING WITH FIXED FEATURE ENGINEERING")
print("="*60)

# Test with the properly engineered features
ybt_predictions_fixed = []
ybt_probs_fixed = []
ybt_targets_fixed = []

x_ybt_tensor_fixed = torch.FloatTensor(x_ybt_imb_scaled_final)
ybt_dataset_fixed = torch.utils.data.TensorDataset(
    x_ybt_tensor_fixed, torch.LongTensor(y_ybt_imb), torch.ones(len(x_ybt_imb_scaled_final))
)
ybt_dataloader_fixed = torch.utils.data.DataLoader(ybt_dataset_fixed, batch_size=32, shuffle=False)

device = next(balanced_model.parameters()).device

with torch.no_grad():
    for batch in ybt_dataloader_fixed:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = balanced_model(x)
        ybt_probs_fixed.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_fixed.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_fixed.extend(y.cpu().numpy())

ybt_probs_fixed = np.array(ybt_probs_fixed)
ybt_predictions_fixed = np.array(ybt_predictions_fixed)
ybt_targets_fixed = np.array(ybt_targets_fixed)

# Calculate metrics
fixed_f1 = f1_score(ybt_targets_fixed, ybt_predictions_fixed, average='weighted')
fixed_auc = roc_auc_score(ybt_targets_fixed, ybt_probs_fixed)

print(f"Balanced DANN on imbalanced YBT (FIXED):")
print(f"F1 Score: {fixed_f1:.3f}")
print(f"ROC-AUC: {fixed_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_fixed)}")
print(f"Prediction distribution: {np.bincount(ybt_predictions_fixed)}")
print(f"Features used: {len(final_features)}")

In [ ]:
print("="*60)
print("VERIFYING THE SUSPICIOUS RESULT")
print("="*60)

# Check the confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix with optimal threshold:")
cm = confusion_matrix(ybt_targets_clipped, ybt_predictions_optimal)
print(cm)

print("\nClassification Report:")
print(classification_report(ybt_targets_clipped, ybt_predictions_optimal))

# Check how many positive samples we're actually detecting
true_positives = cm[1, 1]  # Predicted 1, actually 1
false_negatives = cm[1, 0]  # Predicted 0, actually 1
total_positives = true_positives + false_negatives

print(f"\nPositive class detection:")
print(f"True positives: {true_positives}")
print(f"False negatives: {false_negatives}")
print(f"Total positive samples: {total_positives}")
print(f"Detection rate: {true_positives/total_positives*100:.1f}%")

# Check if this makes sense
print(f"\nThis means we're detecting {true_positives} out of {total_positives} positive cases")
print(f"Which is only {true_positives/total_positives*100:.1f}% of positive cases")

# The high F1 might be due to class imbalance - let's check precision and recall
precision = true_positives / (true_positives + cm[0, 1]) if (true_positives + cm[0, 1]) > 0 else 0
recall = true_positives / total_positives if total_positives > 0 else 0

print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0:.3f}")

# 10. test imbalanced DANN on Imbalanced data 

In [ ]:
print("="*60)
print("CREATING EXACT ARCHITECTURE FROM CHECKPOINT")
print("="*60)

# Based on the checkpoint keys, create the exact architecture
class ImbalancedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim=49, dropout_rate=0.3, learning_rate=0.0005, alpha=1.0, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor: 49 -> 128 -> 64 -> 32
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 128),  # feature_extractor.0.weight: [128, 49]
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),         # feature_extractor.3.weight: [64, 128]
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32)           # feature_extractor.6.weight: [32, 64]
        )
        
        # Classifier: 32 -> 1 (direct, no hidden layers)
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),           # classifier.0.weight: [1, 32]
            nn.Sigmoid()
        )
        
        # Domain discriminator: 32 -> 64 -> 32 -> 1
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(32, 64),          # domain_discriminator.1.weight: [64, 32]
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),          # domain_discriminator.4.weight: [32, 64]
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),           # domain_discriminator.7.weight: [1, 32]
            nn.Sigmoid()
        )
        
        self.domain_weight = domain_weight
        self.criterion = nn.BCELoss()
    
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        
        class_loss = self.criterion(class_output.squeeze(), y.float())
        domain_loss = self.criterion(domain_output.squeeze(), domain.float())
        
        total_loss = class_loss + self.domain_weight * domain_loss
        
        self.log('train_class_loss', class_loss)
        self.log('train_domain_loss', domain_loss)
        self.log('train_total_loss', total_loss)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        
        class_loss = self.criterion(class_output.squeeze(), y.float())
        domain_loss = self.criterion(domain_output.squeeze(), domain.float())
        
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Calculate F1 score
        predictions = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu().numpy(), predictions.cpu().numpy(), average='weighted')
        
        self.log('val_class_loss', class_loss)
        self.log('val_domain_loss', domain_loss)
        self.log('val_total_loss', total_loss)
        self.log('val_f1', f1)
        
        return total_loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_f1"
            }
        }

# Load the model with the exact architecture
imb_model = ImbalancedDomainAdversarialClassifier.load_from_checkpoint(
    '/Users/eb2007/playground/bullpy/c4_play2/lightning_logs/version_14/checkpoints/epoch=21-step=417406.ckpt'
)
imb_model.eval()

# Test on imbalanced YBT data using the 49 features
ybt_predictions_imb = []
ybt_probs_imb = []
ybt_targets_imb = []

# Use the original imbalanced data (49 features)
ybt_dataset_49 = torch.utils.data.TensorDataset(
    torch.FloatTensor(x_ybt_imb_scaled),  # Original 49 features
    torch.LongTensor(ybt_imbalanced['autism_target'].values),
    torch.zeros(len(x_ybt_imb_scaled))
)
ybt_dataloader_49 = torch.utils.data.DataLoader(ybt_dataset_49, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in ybt_dataloader_49:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = imb_model(x)
        ybt_probs_imb.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_imb.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_imb.extend(y.cpu().numpy())

ybt_probs_imb = np.array(ybt_probs_imb)
ybt_predictions_imb = np.array(ybt_predictions_imb)
ybt_targets_imb = np.array(ybt_targets_imb)

# Calculate metrics
imb_on_imb_f1 = f1_score(ybt_targets_imb, ybt_predictions_imb, average='weighted')
imb_on_imb_auc = roc_auc_score(ybt_targets_imb, ybt_probs_imb)

print(f"Imbalanced DANN on imbalanced YBT:")
print(f"F1 Score: {imb_on_imb_f1:.3f}")
print(f"ROC-AUC: {imb_on_imb_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_imb)}")
print(f"Prediction distribution: {np.bincount(ybt_predictions_imb)}")
print(f"Features used: {x_ybt_imb_scaled.shape[1]} (should be 49)")

In [ ]:
print("="*60)
print("LOADING IMBALANCED DANN WITH CORRECT ARCHITECTURE")
print("="*60)

# First, let's check what architecture the checkpoint expects
import torch

checkpoint = torch.load('/Users/eb2007/playground/bullpy/c4_play2/lightning_logs/version_14/checkpoints/epoch=21-step=417406.ckpt', map_location='cpu')
print("Checkpoint keys:")
for key in checkpoint['state_dict'].keys():
    if 'weight' in key:
        print(f"{key}: {checkpoint['state_dict'][key].shape}")

# Create a model with the same architecture as the checkpoint
# Based on the error, it seems the imbalanced model has different layer sizes
class ImbalancedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim=49, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, alpha=1.0, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor (matches the checkpoint architecture)
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = dim
        
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier (simpler architecture based on checkpoint)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        # Domain discriminator (matches checkpoint)
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        self.domain_weight = domain_weight
        self.criterion = nn.BCELoss()
    
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        
        class_loss = self.criterion(class_output.squeeze(), y.float())
        domain_loss = self.criterion(domain_output.squeeze(), domain.float())
        
        total_loss = class_loss + self.domain_weight * domain_loss
        
        self.log('train_class_loss', class_loss)
        self.log('train_domain_loss', domain_loss)
        self.log('train_total_loss', total_loss)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        
        class_loss = self.criterion(class_output.squeeze(), y.float())
        domain_loss = self.criterion(domain_output.squeeze(), domain.float())
        
        total_loss = class_loss + self.domain_weight * domain_loss
        
        # Calculate F1 score
        predictions = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu().numpy(), predictions.cpu().numpy(), average='weighted')
        
        self.log('val_class_loss', class_loss)
        self.log('val_domain_loss', domain_loss)
        self.log('val_total_loss', total_loss)
        self.log('val_f1', f1)
        
        return total_loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_f1"
            }
        }

# Load the model with the correct architecture
imb_model = ImbalancedDomainAdversarialClassifier.load_from_checkpoint(
    '/Users/eb2007/playground/bullpy/c4_play2/lightning_logs/version_14/checkpoints/epoch=21-step=417406.ckpt'
)
imb_model.eval()

# Test on imbalanced YBT data using the 49 features (original imbalanced data)
ybt_predictions_imb = []
ybt_probs_imb = []
ybt_targets_imb = []

# Use the original imbalanced data (49 features)
ybt_dataset_49 = torch.utils.data.TensorDataset(
    torch.FloatTensor(x_ybt_imb_scaled),  # Original 49 features
    torch.LongTensor(ybt_imbalanced['autism_target'].values),
    torch.zeros(len(x_ybt_imb_scaled))
)
ybt_dataloader_49 = torch.utils.data.DataLoader(ybt_dataset_49, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in ybt_dataloader_49:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = imb_model(x)
        ybt_probs_imb.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions_imb.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets_imb.extend(y.cpu().numpy())

ybt_probs_imb = np.array(ybt_probs_imb)
ybt_predictions_imb = np.array(ybt_predictions_imb)
ybt_targets_imb = np.array(ybt_targets_imb)

# Calculate metrics
imb_on_imb_f1 = f1_score(ybt_targets_imb, ybt_predictions_imb, average='weighted')
imb_on_imb_auc = roc_auc_score(ybt_targets_imb, ybt_probs_imb)

print(f"Imbalanced DANN on imbalanced YBT:")
print(f"F1 Score: {imb_on_imb_f1:.3f}")
print(f"ROC-AUC: {imb_on_imb_auc:.3f}")
print(f"Class distribution: {np.bincount(ybt_targets_imb)}")
print(f"Prediction distribution: {np.bincount(ybt_predictions_imb)}")
print(f"Features used: {x_ybt_imb_scaled.shape[1]} (should be 49)")

# 11. comprehensive analysis of all results 

In [ ]:
print("="*60)
print("COMPREHENSIVE COMPARISON OF ALL RESULTS")
print("="*60)

# Results summary
results = {
    "Balanced DANN on Balanced YBT": {"F1": 0.729, "AUC": 0.787},
    "Balanced DANN on Imbalanced YBT": {"F1": 0.006, "AUC": 0.754},
    "Imbalanced DANN on Imbalanced YBT": {"F1": 0.941, "AUC": 0.350}
}

print("PERFORMANCE COMPARISON:")
print("-" * 50)
for model_name, metrics in results.items():
    print(f"{model_name}:")
    print(f"  F1 Score: {metrics['F1']:.3f}")
    print(f"  ROC-AUC: {metrics['AUC']:.3f}")
    print()

print("KEY INSIGHTS:")
print("-" * 50)
print("1. BALANCED DANN on BALANCED data: Good performance (F1=0.729, AUC=0.787)")
print("2. BALANCED DANN on IMBALANCED data: Poor F1 (0.006) but decent AUC (0.754)")
print("3. IMBALANCED DANN on IMBALANCED data: Excellent F1 (0.941) but poor AUC (0.350)")

print("\nINTERPRETATION:")
print("-" * 50)
print("• The imbalanced DANN trained on imbalanced data performs much better on imbalanced data")
print("• The balanced DANN struggles with imbalanced data (threshold issues)")
print("• The low AUC (0.350) for imbalanced DANN suggests it's not discriminating well")
print("• The high F1 (0.941) with low AUC suggests the model is biased toward the majority class")

# Let's check the prediction confidence distribution
print("\n" + "="*60)
print("ANALYZING PREDICTION CONFIDENCE")
print("="*60)

# Calculate prediction confidence
confidence_scores = np.abs(ybt_probs_imb - 0.5) * 2  # Convert to 0-1 scale
print(f"Confidence distribution:")
print(f"  Mean confidence: {np.mean(confidence_scores):.3f}")
print(f"  Std confidence: {np.std(confidence_scores):.3f}")
print(f"  High confidence predictions (>0.8): {np.sum(confidence_scores > 0.8)}")
print(f"  Low confidence predictions (<0.2): {np.sum(confidence_scores < 0.2)}")

# Check if the model is overconfident
print(f"\nPrediction bias analysis:")
print(f"  True positive rate: {np.sum((ybt_predictions_imb == 1) & (ybt_targets_imb == 1)) / np.sum(ybt_targets_imb == 1):.3f}")
print(f"  False positive rate: {np.sum((ybt_predictions_imb == 1) & (ybt_targets_imb == 0)) / np.sum(ybt_targets_imb == 0):.3f}")
print(f"  Precision: {np.sum((ybt_predictions_imb == 1) & (ybt_targets_imb == 1)) / np.sum(ybt_predictions_imb == 1):.3f}")
print(f"  Recall: {np.sum((ybt_predictions_imb == 1) & (ybt_targets_imb == 1)) / np.sum(ybt_targets_imb == 1):.3f}")